## Preliminary Data Analysis - Milestone 1

In [1]:
import numpy as np
import pandas as pd
import os

In [2]:
# load datasets

# all datasets are keyed by ISO alpha-3 codes

data_dir = "../data"

iso_codes = pd.read_csv(f"{data_dir}/iso/countries.csv", index_col="alpha-3")

worldbank_header_size = 4

gdp = pd.read_csv(f"{data_dir}/worldbank/gdp-current-usd-2026.csv", skiprows=worldbank_header_size, index_col="Country Code")
gdp_ppp = pd.read_csv(f"{data_dir}/worldbank/gdp-ppp-international-usd-2021.csv", skiprows=worldbank_header_size, index_col="Country Code")
gdp_pc = pd.read_csv(f"{data_dir}/worldbank/gdp-capita-current-usd-2026.csv", skiprows=worldbank_header_size, index_col="Country Code")
gdp_pc_ppp = pd.read_csv(f"{data_dir}/worldbank/gdp-capita-ppp-international-usd-2021.csv", skiprows=worldbank_header_size, index_col="Country Code")

# drop 2025 from all datasets since the data is missing
gdp = gdp.drop(columns=["2025"])
gdp_ppp = gdp_ppp.drop(columns=["2025"])
gdp_pc = gdp_pc.drop(columns=["2025"])
gdp_pc_ppp = gdp_pc_ppp.drop(columns=["2025"])

In [3]:
# top 25 countries by GDP in 2024
top_gdp_2024 = gdp[gdp.index.isin(iso_codes.index)].nlargest(25, "2024")

def get_oldest_year_with_data(df):
    nonempty_cols = df.columns[df.notna().all()]
    numeric_cols = nonempty_cols[nonempty_cols.str.isdigit()]
    if len(numeric_cols) == 0:
        return None
    return numeric_cols.min()

# oldest year from which we have contiguous data for the 25
oldest_year = get_oldest_year_with_data(top_gdp_2024)

assert(oldest_year == "1990")
assert(pd.isna(gdp.loc["POL", str(int(oldest_year) - 1)])) # pol is available from 1990
assert(pd.isna(gdp.loc["RUS", str(int(oldest_year) - 3)])) # rus is available from 1988

In [4]:
# for top countries, what are the oldest years for which we have data for all 4 datasets?

datasets = {"gdp": gdp, "gdp_ppp": gdp_ppp, "gdp_pc": gdp_pc, "gdp_pc_ppp": gdp_pc_ppp}

# map to oldest year after filtering to the top 25
oldest_years = {name: get_oldest_year_with_data(ds.loc[top_gdp_2024.index]) for name, ds in datasets.items()}

# all data happens to be available from 1990!
assert(all(year == oldest_year for year in oldest_years.values()))

oldest_years

{'gdp': '1990', 'gdp_ppp': '1990', 'gdp_pc': '1990', 'gdp_pc_ppp': '1990'}

## NASDAQ Data

In [5]:
# import symbol metadata

symbols = pd.read_csv(f"{data_dir}/nasdaq/symbols-valid-meta.csv", index_col="Symbol")

etf_path = f"{data_dir}/nasdaq/etf"

# get every csv from etf_path and load it into a dict of dataframes keyed by symbol
etf_dfs = {}
for filename in os.listdir(etf_path):
    if filename.endswith(".csv"):
        symbol = filename[:-4] # basename 
        df = pd.read_csv(f"{etf_path}/{filename}", index_col="Date", parse_dates=True)
        etf_dfs[symbol] = df

In [6]:
# starting dates for each symbol
start_dates = pd.DataFrame({    
    "Symbol": list(etf_dfs.keys()),
    "Start Date": [df.index.min() for df in etf_dfs.values()],
    "Name" : [symbols.loc[symbol, "Security Name"] if symbol in symbols.index else None for symbol in etf_dfs.keys()]
}).set_index("Symbol")

# stupidly extract the relevant country
# by taking the word after MSCI in the name, if it exists
def extract_country(name):
    if pd.isna(name):
        return None
    parts = name.split()
    if "MSCI" in parts:
        idx = parts.index("MSCI")
        if idx + 1 < len(parts):
            return parts[idx + 1]
    return None

start_dates["Country"] = start_dates["Name"].apply(extract_country)

start_dates.sort_values("Start Date").head(25)

,Start Date,Name,Country
Symbol,,,
EWD,1996-03-18,iShares Inc iShares MSCI Sweden ETF,Sweden
EWN,1996-03-18,iShares MSCI Netherlands Index Fund,Netherlands
EWO,1996-03-18,iShares Inc iShares MSCI Austria ETF,Austria
EWM,1996-03-18,iShares MSCI Malaysia Index Fund,Malaysia
EWW,1996-03-18,iShares Inc iShares MSCI Mexico ETF,Mexico
EWK,1996-03-18,iShares Inc iShares MSCI Belgium ETF,Belgium
EWJ,1996-03-18,iShares MSCI Japan Index Fund,Japan
EWS,1996-03-18,iShares Inc iShares MSCI Singapore ETF,Singapore
EWA,1996-03-18,iShares MSCI Australia Index Fund,Australia


## Stock Listing Data (General)

Checking the larger data set (AMEX/NYSE/NASDAQ) and combining it with stock listing data from NASDAQ.

In [7]:
# load the symbol data from NASDAQ

nasdaq_only_symbols = pd.read_csv(f"{data_dir}/stock/nasdaq-listed.csv", index_col="Symbol", sep="|") 
other_listed_symbols = pd.read_csv(f"{data_dir}/stock/other-listed.csv", index_col="NASDAQ Symbol", sep="|")

# clean the metadata from both datasets
nan_symbols_nasdaq = nasdaq_only_symbols[nasdaq_only_symbols.index.str.startswith("File Creation Time")]
assert(len(nan_symbols_nasdaq) == 1)

nan_symbols_other = other_listed_symbols[other_listed_symbols.index.isna()]
assert(len(nan_symbols_other) == 1)
assert(nan_symbols_other["ACT Symbol"].apply(lambda s: s.startswith("File Creation Time")).all())

# drop the rows with the nan symbols
nasdaq_only_symbols = nasdaq_only_symbols.drop(index=nan_symbols_nasdaq.index)
other_listed_symbols = other_listed_symbols.drop(index=nan_symbols_other.index)

# are there any symbols that are in both datasets?
overlap_symbols = set(nasdaq_only_symbols.index).intersection(set(other_listed_symbols.index))
assert(len(overlap_symbols) == 0)

In [8]:
# preprocess and merge the listings

nasdaq_only_symbols["Exchange"] = "N"

# we only care about the following columns
## NASDAQ: Symbol [index], Security Name, Exchange, ETF
## Other: NASDAQ Symbol [index], Security Name, Exchange, ETF

# TODO/FIXME: how is market category defined? we only have this data 
# for NASDAQ, but maybe it's less relevant for ETFs regardless

nasdaq_chosen = nasdaq_only_symbols[["Security Name", "Exchange", "ETF"]]
other_chosen = other_listed_symbols[["Security Name", "Exchange", "ETF"]]

listings = pd.concat([nasdaq_chosen, other_chosen], axis=0)

# check for no data being na in any column
assert(not listings.isna().any().any())

# turn ETF N/Y into boolean
listings["ETF"] = listings["ETF"].map({"Y": True, "N": False})

print(f"Unique exchanges: {listings['Exchange'].unique()}")

listings.head()

Unique exchanges: <StringArray>
['N', 'P', 'Z', 'A', 'M', 'V']
Length: 6, dtype: str


,Security Name,Exchange,ETF
AACB,Artius II Acquisition Inc. - Class A Ordinary ...,N,False
AACBR,Artius II Acquisition Inc. - Rights,N,False
AACBU,Artius II Acquisition Inc. - Units,N,False
AACG,ATA Creativity Global - American Depositary Sh...,N,False
AACIU,Armada Acquisition Corp. III - Units,N,False


In [9]:
# select only the ETFs
etf_listings = listings[listings["ETF"]]

def symbol_to_csv(symbol):
    data_path = f"{data_dir}/stock/history/{symbol}.csv"
    # does the file exist?
    if not os.path.exists(data_path):
        # return nan
        return pd.DataFrame()
    return pd.read_csv(data_path, index_col="date", parse_dates=True)

# for each etf, get the earliest date for which we have data, and its name
etf_start_dates = pd.DataFrame({
    "Symbol": etf_listings.index,
    "Name": etf_listings["Security Name"],
    "Exchange": etf_listings["Exchange"],
    "Start Date": [symbol_to_csv(symbol).index.min() for symbol in etf_listings.index]
}).set_index("Symbol")

print(f"Number of ETFs: {len(etf_start_dates)}")
print(f"Number of ETFs with data: {etf_start_dates['Start Date'].notna().sum()}")

etf_start_dates.sort_values("Start Date").head(10)



Number of ETFs: 5031
Number of ETFs with data: 1741


,Name,Exchange,Start Date
Symbol,,,
VOLT,Tema Electrification ETF,N,1980-03-17
CEF,Sprott Physical Gold and Silver Trust Units,P,1986-04-03
TOT,LionShares U.S. Equity Total Return ETF,P,1991-10-25
SPY,State Street SPDR S&P 500 ETF Trust,P,1993-01-29
ETH,Grayscale Ethereum Staking Mini ETF Shares,P,1993-03-16
MAGS,Roundhill Magnificent Seven ETF,Z,1993-03-23
INTL,Main International ETF,Z,1995-03-06
MDY,State Street SPDR S&P MIDCAP 400 ETF Trust,P,1995-05-04
EWJ,iShares MSCI Japan Index Fund,P,1996-03-18


In [10]:
# repeat the dumb MSCI extraction on this dataset
etf_with_countries = etf_start_dates.copy()
# etf_with_countries["Country"] = etf_with_countries["Name"].apply(extract_country)
# etf_with_countries = etf_with_countries[etf_with_countries["Country"].notna()]

# etf_with_countries.sort_values("Start Date").where(etf_with_countries["Start Date"].notna()).head(30)
etf_with_countries.loc["EWL"]

Name          iShares MSCI Switzerland ETF
Exchange                                 P
Start Date             2020-07-02 00:00:00
Name: EWL, dtype: object

## Market Cap Data

In [11]:

market_cap = pd.read_csv(f"{data_dir}/worldbank/market-cap-current-usd-2026.csv", index_col="Country Code", skiprows=worldbank_header_size)
market_cap = market_cap.drop(columns=["2025"])


In [ ]:
import plotly.graph_objects as go

# availability matrix (1 = data present, 0 = missing) for top-25 GDP countries
year_cols = sorted([c for c in market_cap.columns if c.isdigit()], key=int)
top25_codes = top_gdp_2024.index

market_cap_top25 = market_cap.reindex(top25_codes)
availability = market_cap_top25[year_cols].notna().astype(int)

fig = go.Figure(
    data=go.Heatmap(
        z=availability.values,
        x=year_cols,
        y=availability.index,
        colorscale=[[0.0, "white"], [1.0, "blue"]],
        zmin=0,
        zmax=1,
        showscale=False,
        hovertemplate="Country: %{y}<br>Year: %{x}<br>Status: %{customdata}<extra></extra>",
        customdata=np.where(availability.values == 1, "Data available", "Missing"),
    )
)

fig.update_layout(
    title="Market Cap Data Availability Timeline (Top 25 Countries by 2024 GDP)",
    xaxis_title="Year",
    yaxis_title="Country (ISO-3)",
    xaxis=dict(type="category"),
    template="plotly_white",
    height=700
)

fig.show()